# 1.0 Análisis exploratorio (EDA) - MiBici Guadalajara
**Materia:** Operaciones de aprendizaje automático (TC5061) · Equipo 19

Este notebook es **solo de exploración**. Toda la lógica (limpieza, características, gráficas y modelado)
vive en los módulos de `src/` y aquí únicamente se importa. Las rutas y parámetros salen de `config.yaml`.

In [ ]:
import sys
from pathlib import Path

# Permite importar src/ cuando el proyecto no está instalado con `pip install -e .`
PROJ_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJ_ROOT))

import pandas as pd
from IPython.display import Image, display

from src.config import load_config
from src.dataset import MiBiciDataCleaner
from src.features import MiBiciFeatureEngineer, model_input_columns, temporal_split
from src.plots import generate_eda_figures

config = load_config()
config["data"]["months"]

## 1. Datos brutos y catálogo de estaciones
Generados por `python -m src.dataset`.

In [ ]:
df_raw = pd.read_csv(config.file("raw_dir", "raw_trips_file"))
df_stations = pd.read_csv(config.file("raw_dir", "stations_file"))
print(f"Viajes brutos: {df_raw.shape} | Estaciones en catálogo: {len(df_stations)}")
df_raw.head()

## 2. Calidad de datos: nulos y tipos

In [ ]:
pd.DataFrame({"nulos": df_raw.isna().sum(), "tipo": df_raw.dtypes.astype(str)})

## 3. Limpieza con reglas de negocio (`src/dataset.py`)
Los umbrales vienen de la sección `cleaning` de `config.yaml`.

In [ ]:
cleaner = MiBiciDataCleaner(config)
df_clean = cleaner.clean_frame(df_raw, df_stations)
pd.Series(cleaner.metrics, name="valor")

## 4. Gráficas del EDA (`src/plots.py`)

In [ ]:
for fig_path in generate_eda_figures(df_clean, df_stations, config.path("figures_dir")):
    display(Image(filename=str(fig_path), width=700))

## 5. Matriz de características (`src/features.py`)
Rezagos de 24 h y 168 h, promedio de la misma hora en los 7 días previos y bandera de día festivo.

In [ ]:
df_features = MiBiciFeatureEngineer(config).build_features(df_clean, df_stations)
split = temporal_split(df_features, config)
print(df_features.shape)
print(split.value_counts())
df_features[["Origen_Id", "fecha_hora", "demand_count"] + model_input_columns(config)].head()

In [ ]:
target = config["features"]["target"]
print(f"Demanda media por estación-hora: {df_features[target].mean():.3f}")
print(f"Porcentaje de horas con demanda cero: {(df_features[target] == 0).mean():.1%}")
df_features.groupby(split)[target].mean().rename("demanda_media")

## 6. Resultados de modelado
Generados por `python -m src.modeling.train` (detalle completo en MLflow UI).

In [ ]:
comparison_path = config.path("reports_dir") / "model_comparison.csv"
pd.read_csv(comparison_path) if comparison_path.exists() else "Ejecuta primero: python -m src.modeling.train"